# 01 - Baselines (skill 5)
Honest "before" numbers for Tiny Aya: zero-shot and few-shot on `val`, then the locked test splits
(allowed for **base** models only; every choice is made from `val`). Run on a Colab/Kaggle GPU.
Set the three values in the parameters cell, add `HF_TOKEN` (and `GEMINI_API_KEY` if you run
the API baseline) as Colab secrets, then run top to bottom. The run is resumable: if the
session disconnects, re-run the same cell and finished episodes are skipped.

In [ ]:
# 1. GPU + Python check
!nvidia-smi
import sys
print(sys.version)

In [ ]:
# 2. parameters - edit these
REPO_URL = "https://github.com/YOUR_GITHUB_USER/tinyops"   # your public repo
HF_USER = "YOUR_HF_USERNAME"                                # same one you registered with
DATA_REPO = f"{HF_USER}/tinyops-data"                       # HF dataset repo for results
BATCH_SIZE = 8            # T4: 8, L4: 16
RUN_FEWSHOT_ON_TEST = True
RUN_AYA_EXPANSE = False   # optional 8B baseline (4-bit)

In [ ]:
# 3. clone + install (Python 3.12 is required; make a uv venv if the runtime is older)
import os, subprocess
if not os.path.exists("tinyops"):
    !git clone {REPO_URL} tinyops
%cd tinyops
if sys.version_info >= (3, 12):
    PY = "python"
    !pip install -q -e ".[train]"
else:
    PY = "/content/venv/bin/python"
    !pip -q install uv
    !uv venv --python 3.12 /content/venv
    !uv pip install --python {PY} -q -e ".[train]"

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:  # noqa: BLE001 - Colab raises its own secret-not-found error types
    print("no GEMINI_API_KEY secret (only needed for the API baseline)")
!{PY} -m tinyops.utils.envinfo

In [ ]:
# 4. data is committed in the repo; verify the locked test splits before using anything
!{PY} -m tinyops.scenarios.build_dataset verify

In [ ]:
# 5. Tiny Aya zero-shot on val (180 scenarios)
M = "CohereLabs/tiny-aya-global"
!{PY} -m tinyops.eval.run_eval --policy hf --model {M} --split val --batch-size {BATCH_SIZE} \
    --out results/episodes/tinyaya_zeroshot_val.jsonl

In [ ]:
# 6. Tiny Aya few-shot (k=2, train-split oracle examples) on val
!{PY} -m tinyops.eval.run_eval --policy hf --model {M} --fewshot 2 --split val \
    --batch-size {BATCH_SIZE} --out results/episodes/tinyaya_fewshot_val.jsonl

In [ ]:
# 7. the "before" numbers on the locked test splits (base model only)
for split in ["test", "test_injection", "test_ood"]:
    !{PY} -m tinyops.eval.run_eval --policy hf --model {M} --split {split} --batch-size {BATCH_SIZE} \
        --out results/episodes/tinyaya_zeroshot_{split}.jsonl
    if RUN_FEWSHOT_ON_TEST:
        !{PY} -m tinyops.eval.run_eval --policy hf --model {M} --fewshot 2 --split {split} \
            --batch-size {BATCH_SIZE} --out results/episodes/tinyaya_fewshot_{split}.jsonl

In [ ]:
# 8. optional: Aya Expanse 8B zero-shot on val (4-bit so it fits a T4)
if RUN_AYA_EXPANSE:
    !{PY} -m tinyops.eval.run_eval --policy hf --model CohereLabs/aya-expanse-8b --load-in-4bit \
        --split val --batch-size 4 --out results/episodes/ayaexpanse8b_zeroshot_val.jsonl

In [ ]:
# 9. failure-mode analysis from the val episodes -> docs/BASELINE_FINDINGS.md
runs = ["tinyaya_zeroshot_val", "tinyaya_fewshot_val"] + (["ayaexpanse8b_zeroshot_val"] if RUN_AYA_EXPANSE else [])
args = " ".join(f"--runs {r}" for r in runs)
!{PY} -m tinyops.eval.baseline_findings --split val {args}
from pathlib import Path
print(Path("docs/BASELINE_FINDINGS.md").read_text(encoding="utf-8")[:3000])

In [ ]:
# 10. push episodes + metrics + manifests to the HF dataset repo (folder baselines/)
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
for sub in ["episodes", "metrics", "runs"]:
    api.upload_folder(folder_path=f"results/{sub}", path_in_repo=f"baselines/{sub}",
                      repo_id=DATA_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj="results/compute_log.csv", path_in_repo="baselines/compute_log.csv",
                repo_id=DATA_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj="docs/BASELINE_FINDINGS.md", path_in_repo="baselines/BASELINE_FINDINGS.md",
                repo_id=DATA_REPO, repo_type="dataset")
print("uploaded to", DATA_REPO)

## Gemini reference (run from the laptop, CPU + API)
Closed-model reference on `test` and `test_injection` (model name and token budget in
`configs/eval.yaml`; see DECISIONS.md for how the name was checked):

```
python -m tinyops.eval.run_eval --policy api --model gemini/gemini-3.6-flash --max-new-tokens 1024 --split test --out results/episodes/gemini_test.jsonl
python -m tinyops.eval.run_eval --policy api --model gemini/gemini-3.6-flash --max-new-tokens 1024 --split test_injection --out results/episodes/gemini_test_injection.jsonl
```

Download the `results/` folder from this notebook (or pull it from the HF dataset repo) and
commit the metrics files; then fill in `docs/BASELINE_FINDINGS.md` by re-running step 9 locally.